# The escalated case: the full pass, alone

**Week 1, Wednesday afternoon, the escalated case.** Anand has replied: "Send the reconciliation
and the log before the day closes. My analyst checks it tonight." Run the whole pass the six
chapters built: read and profile, convert with a rejects log, apply the identity rule, make the
two open decisions, reconcile in rows and in rupees, draw the bridge, and recompute Tuesday.

Each step has lettered choices above a placeholder such as `__TODO1__`. Replace the placeholder
with the code of the option you choose, run the step, and read its checks. Run as shipped, the
notebook stops at the first placeholder with a `NameError`; that is expected. Post your eight
letters in order when every check passes, then write the note to Finance in under 120 words.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

import csv, json, math
from collections import Counter
from datetime import date

DATA = kit.data_dir()
ORDERS_CSV = DATA / "C2_W01_D03_orders_STUDENT.csv"
BOOKS_Q1 = 19000000        # Anand's Q1 to the rupee, which his analyst sends on request

def read_orders(path=ORDERS_CSV):
    """Every row of an export as a dictionary of text, with the file line it came from."""
    with open(path, newline="", encoding="utf-8") as f:
        return [dict(row, line=n) for n, row in enumerate(csv.DictReader(f), start=2)]

def convert(value):
    """A whole number of rupees, or None with the reason, so a failure is counted, never hidden."""
    try:
        return int(value), ""
    except (TypeError, ValueError):
        return None, "amount does not convert to a whole number"

def Q(rows, quarter):
    """Rupees in a quarter over the amounts that convert."""
    return sum(convert(r["amount"])[0] or 0 for r in rows if r["quarter"] == quarter)

def apply_rule(rows, key, prefer):
    """One row per key; prefer is "validates" or "first". Returns (kept, set aside)."""
    kept, aside = {}, []
    for r in rows:
        k = r[key] if isinstance(key, str) and key in r else tuple(sorted(r.items()))
        if k not in kept:
            kept[k] = r
        elif prefer == "validates" and convert(kept[k]["amount"])[0] is None and convert(r["amount"])[0] is not None:
            aside.append(kept[k]); kept[k] = r
        else:
            aside.append(r)
    good = [dict(r, amount=convert(r["amount"])[0]) for r in kept.values() if convert(r["amount"])[0] is not None]
    aside += [r for r in kept.values() if convert(r["amount"])[0] is None]
    return good, aside

def per_customer(rows, q, seg):
    rs = [r for r in rows if r["quarter"] == q and r["segment"] == seg]
    return len(rs) / len({r["customer_id"] for r in rs})

print("helper and case functions loaded")

In [ ]:
kit.side_by_side(
    kit.ladder(["chapters 1 to 6, the pass built", "the escalated case: the full pass alone",
            "the second case: the auditor asks why"], lit=1, show=False),
    kit.vflow(["read and profile", "convert with a log", "the identity rule", "two decisions",
               "reconcile twice", "Tuesday and the note"], lit=0, show=False),
)

## Step 1. Read and profile

In [ ]:
# TODO 1. Which call reads the export so every value can be profiled as the file holds it?
#   a) [dict(r) for r in csv.reader(open(ORDERS_CSV))]
#   b) read_orders()
#   c) json.load(open(ORDERS_CSV))
#   d) open(ORDERS_CSV).read().split(",")
raw = __TODO1__
ids = {r["order_id"] for r in raw}
print("Step 1: done")

In [ ]:
kit.check("201 rows read", len(raw) == 201, f"{len(raw)}")
kit.check("186 distinct order ids", len(ids) == 186, f"{len(ids)}")
kit.bars([("rows", len(raw)), ("distinct order ids", len(ids))], title="Rows against orders in the export")

## Step 2. Convert with a rejects log

In [ ]:
# TODO 2. What should happen to an amount that int() refuses?
#   a) it becomes 0, so the total runs
#   b) the row is deleted before anyone sees it
#   c) it goes to the rejects log with its line and reason
#   d) it becomes the median amount of its segment
failed = [r for r in raw if convert(r["amount"])[0] is None]
rejects_log = __TODO2__
print("Step 2: done")

In [ ]:
kit.check("one amount is logged, not zeroed", len(rejects_log) == 1 and all("line" in x for x in rejects_log))

## Step 3. The identity rule

In [ ]:
# TODO 3. Which key says two rows are the same order?
#   a) the whole record, file line included
#   b) customer_id and amount together
#   c) the whole record, file line excluded
#   d) order_id alone
KEY = __TODO3__

# TODO 4. Inside a group of rows sharing that key, which copy stays?
#   a) the first copy in the file
#   b) the copy whose amount converts
#   c) the copy with the larger amount
#   d) the last copy in the file
PREFER = __TODO4__
clean, set_aside = apply_rule(raw, KEY, PREFER)
print("Step 3: done")

In [ ]:
kit.check("186 orders kept", len(clean) == 186, f"{len(clean)}")
kit.check("15 rows set aside", len(set_aside) == 15, f"{len(set_aside)}")
kit.check("Q1 is Rs 1,90,00,000", Q(clean, "Q1") == 19000000, kit.rupees(Q(clean, "Q1")))

## Step 4. The two open decisions

In [ ]:
# TODO 5. One order has no status. Which decision keeps revenue and the delivered count honest?
#   a) keep and flag
#   b) drop the order
#   c) default it to delivered
#   d) default it to cancelled
STATUS_DECISION = __TODO5__

# TODO 6. The largest Q2 order is 1.66 times the next. What happens to it?
#   a) remove it as an outlier
#   b) cap it at the next largest order
#   c) keep it and flag it, shown with and without
#   d) move it to Q1
BULK_DECISION = __TODO6__
decisions = [("status missing on one Q2 order", STATUS_DECISION), ("largest Q2 order", BULK_DECISION)]
print("Step 4: done")

In [ ]:
kit.check("Q2 stays at Rs 1,87,00,000", Q(clean, "Q2") == 18700000 and "keep" in STATUS_DECISION and "keep" in BULK_DECISION)
kit.columns(["Q1", "Q2"], [("clean, Rs lakh", [Q(clean, "Q1") / 1e5, Q(clean, "Q2") / 1e5])],
            fmt=lambda v: f"{v:,.0f}", title="Clean revenue by quarter")

## Step 5. Reconcile twice and draw the bridge

In [ ]:
# TODO 7. Which pair of checks proves the reconciliation to Anand's analyst?
#   a) rows in equal rows kept, and Q1 rounds to 1.9 crore
#   b) rows in equal kept plus set aside, and exported Q1 less set-aside rupees equals the books
#   c) rows kept equal distinct ids, and Q2 is unchanged
#   d) the rejects log is empty, and Q1 is below the dashboard
exported_q1 = Q(raw, "Q1")
removed_q1 = sum(convert(r["amount"])[0] or 0 for r in set_aside if r["quarter"] == "Q1")
proof = __TODO7__
print("Step 5: done")

In [ ]:
kit.check("rows reconcile", len(raw) == len(clean) + len(set_aside))
kit.check("rupees reconcile to the books", exported_q1 - removed_q1 == 19000000)
kit.bridge(("Q1 as exported", exported_q1), [("rows set aside", -removed_q1)], end_label="Q1 clean, the books",
           lo=18800000, title="Q1 from the export to the books; the axis starts at Rs 1.88 crore")

## Step 6. Tuesday recomputed, and the note

In [ ]:
# TODO 8. Retail-Plus orders per customer fell 49.0 percent as Tuesday reported. On clean data?
#   a) it vanishes
#   b) it grows past 49 percent
#   c) it moves to Retail-Core
#   d) it falls 35.0 percent, smaller
TUESDAY_VERDICT = __TODO8__
rp = [per_customer(clean, q, "Retail-Plus") for q in ("Q1", "Q2")]
print("Step 6: done")

In [ ]:
kit.check("Retail-Plus falls about 35 percent on clean data", round(100 * (rp[1] - rp[0]) / rp[0], 1) == -35.0)
kit.line(["Q1", "Q2"], [("Retail-Plus orders per customer, clean", [round(x, 2) for x in rp], "good"),
                        ("as Tuesday reported", [2.32, 1.18], "bad")], fmt=lambda v: f"{v:.2f}",
         title="Retail-Plus frequency: the finding stands, smaller")
kit.check_summary()

**Post** your eight letters in order, then the note to Finance: numbers first, which figure is right
and why, the two reconciliations, the two flagged decisions, and whether Tuesday's finding survives.